# Эксперимент: последовательности переговоров и контакты перед закрытием
Использует подготовленную матрицу из avito_audit_fixes.zip и новый feature/chat_features_audit.csv. Не перезаписывает исходный submission. Проверка по продавцам, те же три seed, фиксированный порог 0.50. Кандидаты: этапы + старый ансамбль; этапы без старого NLP; новый NLP вместо старых признаков (старые псевдометки сохранены только в этом контрольном сравнении). Повторная валидация на тех же 300 метках не является независимым тестом.

In [1]:
import os
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name in {"analysis", "feature", "models"}:
    repo_root = repo_root.parent

assert (repo_root / "requirements.txt").is_file() and (repo_root / "data/listings.csv").is_file(), (
    "Запустите ноутбук из корня репозитория или из папки, в которой он лежит"
)
os.chdir(repo_root)

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold


In [3]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
EXTENDED_VERIFIED_PATH = Path("analysis/extend_verified.csv")
LISTINGS_PATH = Path("data/listings.csv")
OUTPUT_DIR = Path("models/stage_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
CV_SEEDS = [11, 42, 107]
THRESHOLD = 0.50
CHAT_MODEL_SHARE = 0.75
CHAT_LABEL_WEIGHT = 0.10
CHAT_C = 0.3
VERIFIED_C = 1.0
WEAK_WEIGHT_CAP = 0.03
WEAK_CLASS_MASS = 15.0


In [4]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.is_unique, "Повторяются ID в проверенной разметке"
assert verified.index.isin(features.index).all()
assert verified["is_deal"].isin([0, 1]).all(), "Некорректные метки в verified.csv"
assert features.columns.is_unique

feature_columns = features.columns.difference(["seller_id", "is_deal", "weight"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]
categories = features.loc[y.index, "category"]
assert categories.notna().all(), "Для CatShareMAE нужна категория каждого проверенного объявления"

print(f"Объявлений в матрице: {len(X)}")
print(f"Признаков: {X.shape[1]}")
print(f"Объявлений с проверенными метками: {len(y)}")
assert X.shape == (15274, 139), f"Нужна матрица с 4 interaction-признаками, получено {X.shape}"
assert len(y) == 300
seller_groups = features["seller_id"]
assert seller_groups.notna().all()
blocked_ids = listings.index[listings["close_type"].eq("заблокировано")]


Объявлений в матрице: 15274
Признаков: 139
Объявлений с проверенными метками: 300


In [5]:
interaction_columns = ["strong_chat_signal", "survey_changed_no_strong_chat",
                       "survey_elsewhere_no_strong_chat", "survey_avito_strong_chat"]
legacy_nlp_columns = ["scored_message_count", "strongest_positive_signal", "strongest_negative_signal", "deal_signal_score", "deal_signal_strength"]
assert set(interaction_columns + legacy_nlp_columns).issubset(X.columns)
legacy = pd.read_csv(EXTENDED_VERIFIED_PATH, index_col="listing_id")
assert legacy.index.is_unique
assert legacy.loc[y.index, "is_deal"].eq(y).all()
assert legacy.loc[y.index, "weight"].eq(1).all()
legacy = legacy.loc[legacy["weight"].eq(1) & ~legacy.index.isin(y.index), ["is_deal", "weight"]].assign(weight=CHAT_LABEL_WEIGHT)
assert len(legacy) == 726, f"Ожидали прежние 726 NLP, получили {len(legacy)}"
# Два источника слабой разметки: совместные признаки опроса и переписки.
# Не используем все positive-only survey/NLP как ground truth.
pos = X["survey_avito_strong_chat"].eq(1)
neg = X["survey_changed_no_strong_chat"].eq(1) | X["survey_elsewhere_no_strong_chat"].eq(1)
weak = pd.DataFrame({"is_deal": pos.astype(int)}, index=X.index).loc[pos ^ neg]
weak = weak.loc[~weak.index.isin(y.index) & ~weak.index.isin(blocked_ids)].copy()
assert weak.index.is_unique
assert weak.index.intersection(y.index).empty
assert legacy.index.intersection(y.index).empty
print("Legacy NLP:", legacy["is_deal"].value_counts().to_dict())
print("Weak candidates:", weak["is_deal"].value_counts().to_dict())


Legacy NLP: {1: 726}
Weak candidates: {0: 2454, 1: 1150}


In [6]:
def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, make_column_selector(dtype_include=["number", "bool"])),
    ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=["number", "bool"])),
])
verified_model = make_pipeline(
    preprocessor,
    LogisticRegression(C=VERIFIED_C, max_iter=2000),
)
chat_model = clone(verified_model).set_params(logisticregression__C=CHAT_C)

In [7]:

import re
# Очищенные события/сообщения; интервалы ограничены сроком жизни объявления.
data_dir = Path("data/data_wthout_anomailes")
msg = pd.read_csv(data_dir / "messages_wthout_anomailes.csv")
evt = pd.read_csv(data_dir / "events_wthout_anomailes.csv")
dates = pd.read_csv("data/listings.csv", usecols=["listing_id","published_at","closed_at"]).drop_duplicates("listing_id").set_index("listing_id")
for col in dates:
    dates[col] = pd.to_datetime(dates[col], errors="coerce", format="mixed")
def within_lifetime(df, t):
    df = df.join(dates, on="listing_id")
    df[t] = pd.to_datetime(df[t], errors="coerce", format="mixed")
    return df.loc[df[t].between(df.published_at, df.closed_at)].copy()

msg = within_lifetime(msg, "sent_at").dropna(subset=["text"])
evt = within_lifetime(evt, "event_time")
msg = msg.sort_values(["listing_id","chat_id","sent_at"], kind="stable")
s = msg.text.str.lower().str.replace("ё","е").str.replace(r"[^а-яa-z0-9]+"," ",regex=True).str.strip()
buyer = msg.author.eq("buyer")
seller = msg.author.eq("seller")
# Arrival is separated from intention to travel; only buyer refusals invalidate arrival.
patterns = {
    "intent": (buyer, r"^(?:беру|забираю|отложите для меня|хорошо договорились)\b"),
    "address": (buyer, r"^(?:куда подъехать|скиньте адрес|когда можно подъехать|во сколько удобно забрать)\b"),
    "travel": (buyer, r"^(?:еду|выезжаю|уже подхожу|через \d+ минут буду|извините опаздываю)\b"),
    "arrived": (buyer, r"^(?:подъехал|я на месте)\b"),
    "meet": (seller, r"^(?:спускаюсь|выхожу|сейчас выйду)\b"),
    "thanks": (buyer, r"^(?:спасибо(?: все отлично)?|все супер спасибо)$"),
    "refused": (buyer, r"купил в другом месте|не приеду|не смогу приехать|извините передумал|не возьму|посмотрел не подошло|не то что ожидал|^дорого спасибо$"),
    "sold": (seller, r"^(?:продано|уже продано|извините уже продал|уже нет извините)$"),
    "waiting": (seller, r"^(?:вы приедете|вы где|ждать вас)\b"),
}
keys = ["listing_id","chat_id"]
g = msg.groupby(keys)
ch = g.agg(last=("sent_at","max"), first=("sent_at","min"), close=("closed_at","first"), n=("text","size"))
for name, (author, pattern) in patterns.items():
    mask = author & s.str.contains(pattern, regex=True, na=False)
    ch[name] = msg.loc[mask].groupby(keys).sent_at.max()
arr = ch.arrived.notna() & ~ch.refused.gt(ch.arrived)
ch["actual_arrival"] = arr
ch["confirmed_meet"] = arr & ch.meet.ge(ch.arrived)
ch["after_arrival_thanks"] = arr & ch.thanks.gt(ch.arrived)
ch["failed_arrival"] = ch.arrived.notna() & ch.refused.gt(ch.arrived)
ch["travel_no_arrival"] = ch.travel.notna() & ch.arrived.isna()
ch["intent_no_arrival"] = ch.intent.notna() & ch.arrived.isna()
ch["late_refusal"] = ch.refused.notna() & (ch.intent.notna() | ch.travel.notna() | ch.arrived.notna())
ch["intent_or_address"] = ch.intent.notna() | ch.address.notna()
ch["unanswered_wait"] = ch.waiting.notna() & ch.waiting.eq(ch["last"])
# Сделка может случиться в одном чате среди множества неудачных.
stage = pd.DataFrame(index=X.index)
for name in ["actual_arrival","confirmed_meet","after_arrival_thanks","failed_arrival","travel_no_arrival","intent_no_arrival","late_refusal","intent_or_address","unanswered_wait"]:
    stage["stage_"+name+"_count"] = ch[name].groupby(level=0).sum().reindex(X.index,fill_value=0)
    stage["stage_"+name+"_share"] = ch[name].groupby(level=0).mean().reindex(X.index,fill_value=0)
for name in ["arrived","meet","thanks","refused","sold","travel"]:
    gap = (ch.close - ch[name]).dt.total_seconds()/3600
    stage["stage_"+name+"_close_loghours"] = np.log1p(gap.groupby(level=0).min().reindex(X.index))
    stage["stage_"+name+"_last24h"] = gap.le(24).groupby(level=0).sum().reindex(X.index,fill_value=0)
# Отдельный канал телефонных контактов: время до закрытия, количество, динамика.
phone = evt.loc[evt.event_type.eq("phone_view")].copy()
phone["gap"] = (phone.closed_at-phone.event_time).dt.total_seconds()/3600
stage["phone_last_loghours"] = np.log1p(phone.groupby("listing_id").gap.min().reindex(X.index))
for hours in [2, 12, 24, 48]:
    stage[f"phone_last_{hours}h"] = phone.loc[phone.gap.le(hours)].groupby("listing_id").size().reindex(X.index,fill_value=0)
# Новые NLP-оценки отдельно для контролируемой абляции.
audit = pd.read_csv("feature/chat_features_audit.csv")
assert audit.chat_id.is_unique
audit_ag = audit.groupby("listing_id").agg(
    audit_pos=("strongest_positive_signal","max"),
    audit_neg=("strongest_negative_signal","max"),
    audit_net_max=("deal_signal_score","max"),
    audit_net_min=("deal_signal_score","min"),
    audit_pos_count=("deal_signal_score",lambda t:t.gt(0).sum()),
    audit_neg_count=("deal_signal_score",lambda t:t.lt(0).sum()),
).reindex(X.index).fillna(0)
print("Stage/event features:",stage.shape, "Audit NLP:",audit_ag.shape)


Stage/event features: (15274, 35) Audit NLP: (15274, 6)


In [8]:
CANDIDATES = {
    "05_stages_legacy": (X.join(stage), "legacy"),
    "06_stages_no_legacy": (X.drop(columns=legacy_nlp_columns).join(stage), "none"),
    "07_audit_nlp": (X.drop(columns=legacy_nlp_columns).join(audit_ag), "legacy"),
}


def pseudo_for(mode, heldout_sellers):
    if mode == "none":
        return legacy.iloc[:0].copy()
    pool = (legacy if mode == "legacy" else weak).copy()
    pool = pool.loc[~seller_groups.loc[pool.index].isin(heldout_sellers)]
    if mode == "weak":
        counts = pool["is_deal"].value_counts()
        if len(counts) != 2:
            return legacy.iloc[:0].copy()
        # Одинаковый суммарный вес каждого класса, ограничиваем вес одной метки.
        mass = min(WEAK_CLASS_MASS, WEAK_WEIGHT_CAP * counts.min())
        pool["weight"] = pool["is_deal"].map(mass / counts)
    assert pool.index.intersection(y.index).empty
    return pool


def fit_one(estimator, frame, target, pseudo):
    labels = pd.concat([target.to_frame().assign(weight=1.0), pseudo])
    assert labels.index.is_unique
    return clone(estimator).fit(frame.loc[labels.index], labels["is_deal"],
        logisticregression__sample_weight=labels["weight"])


def predict(frame, mode, target, test_ids, heldout_sellers):
    empty = legacy.iloc[:0]
    gold = fit_one(verified_model, frame, target, empty)
    p = gold.predict_proba(frame.loc[test_ids])[:, 1]
    if mode != "none":
        pseudo = pseudo_for(mode, heldout_sellers)
        augmented = fit_one(chat_model, frame, target, pseudo)
        p = CHAT_MODEL_SHARE * augmented.predict_proba(frame.loc[test_ids])[:, 1] + (1 - CHAT_MODEL_SHARE) * p
    p[pd.Index(test_ids).isin(blocked_ids)] = 0.0
    return p


In [9]:
def competition_score(f1, cat_share_mae):
    f1_points = 20 * np.clip((f1 - 0.60) / 0.25, 0, 1)
    share_points = 10 * np.clip((15 - cat_share_mae) / 14, 0, 1)
    return f1_points + share_points


In [10]:
# Общие разбиения: ни один продавец из gold test не входит в gold train или псевдометки.
results = []
oof_records = []
for seed in CV_SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    folds = list(cv.split(X.loc[y.index], y, groups=seller_groups.loc[y.index]))
    for name, (frame, mode) in CANDIDATES.items():
        oof = pd.Series(np.nan, index=y.index)
        for train, test in folds:
            target = y.iloc[train]
            test_ids = y.index[test]
            heldout = set(seller_groups.loc[test_ids])
            assert not set(seller_groups.loc[target.index]) & heldout
            oof.loc[test_ids] = predict(frame, mode, target, test_ids, heldout)
        assert oof.notna().all()
        pred = (oof > THRESHOLD).astype(int)
        shares = pd.DataFrame({"actual": y, "pred": pred, "category": categories}).groupby("category")[["actual", "pred"]].mean()
        mae = 100 * (shares["actual"] - shares["pred"]).abs().mean()
        f1 = f1_score(y, pred, zero_division=0)
        results.append({"candidate": name, "seed": seed, "F1": f1, "CatShareMAE": mae,
                        "Precision": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred, zero_division=0), "Score": competition_score(f1, mae)})
        oof_records.append(pd.DataFrame({"candidate": name, "seed": seed, "listing_id": y.index, "actual": y.to_numpy(), "probability": oof.to_numpy(), "prediction": pred.to_numpy()}))
        print(name, seed, f"F1={f1:.4f} Score={competition_score(f1, mae):.4f}", flush=True)
results = pd.DataFrame(results)
display(results.round(4))
display(results.groupby("candidate").mean(numeric_only=True).drop(columns="seed").round(4))
results.to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
pd.concat(oof_records, ignore_index=True).to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)


05_stages_legacy 11 F1=0.8609 Score=27.8581
06_stages_no_legacy 11 F1=0.8235 Score=25.8619
07_audit_nlp 11 F1=0.8240 Score=26.2287
05_stages_legacy 42 F1=0.8793 Score=29.0809
06_stages_no_legacy 42 F1=0.8384 Score=26.9136
07_audit_nlp 42 F1=0.8235 Score=25.6999
05_stages_legacy 107 F1=0.8511 Score=29.3239
06_stages_no_legacy 107 F1=0.8106 Score=24.5014
07_audit_nlp 107 F1=0.8270 Score=26.4258


,candidate,seed,F1,CatShareMAE,Precision,Recall,Score
0,05_stages_legacy,11,0.8609,3.9987,0.8684,0.8534,27.8581
1,06_stages_no_legacy,11,0.8235,3.8287,0.8667,0.7845,25.8619
2,07_audit_nlp,11,0.8240,3.3716,0.8205,0.8276,26.2287
3,05_stages_legacy,42,0.8793,2.2867,0.8793,0.8793,29.0809
4,06_stages_no_legacy,42,0.8384,4.0249,0.8496,0.8276,26.9136
5,07_audit_nlp,42,0.8235,4.0554,0.8033,0.8448,25.6999
6,05_stages_legacy,107,0.8511,1.9466,0.8403,0.8621,29.3239
7,06_stages_no_legacy,107,0.8106,4.2822,0.8288,0.7931,24.5014
8,07_audit_nlp,107,0.8270,3.4283,0.8099,0.8448,26.4258


,F1,CatShareMAE,Precision,Recall,Score
candidate,,,,,
05_stages_legacy,0.8637,2.7440,0.8627,0.8649,28.7543
06_stages_no_legacy,0.8242,4.0453,0.8484,0.8017,25.7590
07_audit_nlp,0.8249,3.6184,0.8112,0.8391,26.1182


In [11]:
# Сохраняем отдельные кандидаты. Никакого автоматического выбора по тем же 300 меткам.
for name, (frame, mode) in CANDIDATES.items():
    p = predict(frame, mode, y, frame.index, set())
    submission = pd.DataFrame({"listing_id": frame.index, "is_deal": (p > THRESHOLD).astype(int)})
    assert submission["listing_id"].is_unique
    assert set(submission["listing_id"]) == set(listings.index)
    assert submission["is_deal"].isin([0, 1]).all()
    assert not submission.loc[submission["listing_id"].isin(blocked_ids), "is_deal"].any()
    path = OUTPUT_DIR / f"submission_{name}.csv"
    submission.to_csv(path, index=False)
    print(path, "positive:", submission["is_deal"].sum())


models\stage_outputs\submission_05_stages_legacy.csv positive: 5839
models\stage_outputs\submission_06_stages_no_legacy.csv positive: 5558
models\stage_outputs\submission_07_audit_nlp.csv positive: 6045
